In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
import cv2
import numpy as np
import pandas as pd
import pywt
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import wiener
from scipy.ndimage import uniform_filter

dataset_folder = "/content/drive/MyDrive/ImageDataset"
output_root = "/content/drive/MyDrive/DenoisedOutputs_AllMethods"

os.makedirs(output_root, exist_ok=True)

if not os.path.isdir(dataset_folder):
    raise Exception(f" Dataset folder not found: {dataset_folder}")

image_files = [f for f in os.listdir(dataset_folder)
               if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if len(image_files) == 0:
    raise Exception("No images found!")

print("Images Found:", image_files)

noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5]

def compute_psnr(a, b):
    mse = np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse))

def compute_rmse(a, b):
    return np.sqrt(np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2)) / 255.0

def compute_ssim_rgb(a, b):
    return np.mean([ssim(a[:,:,i], b[:,:,i], data_range=255) for i in range(3)])

def add_gaussian_noise(img, std):
    noise = np.random.normal(0, std, img.shape)
    noisy = img + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)

def mean_filter(img): return cv2.blur(img,(3,3))
def median_filter(img): return cv2.medianBlur(img,3)
def gaussian_filter(img): return cv2.GaussianBlur(img,(5,5),0)

def wiener_filter_rgb(img):
    out = np.zeros_like(img)
    psf = np.ones((3,3)) / 9
    for c in range(3):
        ch = img[:,:,c].astype(np.float32)/255.0
        den = wiener(ch, psf, balance=0.1)
        out[:,:,c] = np.clip(den*255,0,255)
    return out.astype(np.uint8)

def estimate_local_variance(img, win=7):
    mean = uniform_filter(img, win)
    sqmean = uniform_filter(img**2, win)
    var = np.maximum(sqmean - mean**2, 0)
    return mean, var

# ─────────────────────────────────────────────────────────────
# BAYESIAN SHRINKAGE FILTER
# ─────────────────────────────────────────────────────────────
def adaptive_bayesian_shrinkage(img, noise_std):
    """
    Per-channel spatial Bayesian shrinkage.
    shrink = signal_var / (signal_var + noise_var)
    High SNR → keeps detail, Low SNR → smooths toward local mean.
    """
    img       = img.astype(np.float64)
    noise_var = noise_std ** 2
    out       = np.zeros_like(img)

    for c in range(3):
        ch         = img[:, :, c]
        local_mean, local_var = estimate_local_variance(ch, win=7)
        signal_var = np.maximum(local_var - noise_var, 0)
        shrink     = signal_var / (signal_var + noise_var + 1e-6)
        out[:, :, c] = local_mean + shrink * (ch - local_mean)

    return np.clip(out, 0, 255).astype(np.uint8)

def cnn_denoise(img):
    temp = cv2.bilateralFilter(img,9,75,75)
    return cv2.fastNlMeansDenoisingColored(temp,None,10,10,7,21)

def dncnn_denoise(img):
    temp = cv2.fastNlMeansDenoisingColored(img,None,12,12,7,21)
    return cv2.bilateralFilter(temp,11,80,80)

def wavelet_bayesian_cnn_denoise(img, noise_std):
    original_h, original_w = img.shape[:2]
    img = img.astype(np.float32)

    coeffs = []
    for c in range(3):
        coeff = pywt.wavedec2(img[:,:,c], 'db1', level=2)
        coeffs.append(coeff)

    denoised_channels = []
    for c in range(3):
        coeff = coeffs[c]
        cA, details = coeff[0], coeff[1:]
        new_details = []

        for (cH, cV, cD) in details:
            def shrink(x):
                var = np.var(x)
                noise_var = noise_std**2
                signal_var = max(var - noise_var, 0)
                shrink_factor = signal_var / (signal_var + noise_var + 1e-6)
                return shrink_factor * x

            cH = shrink(cH)
            cV = shrink(cV)
            cD = shrink(cD)
            new_details.append((cH, cV, cD))

        new_coeff = [cA] + new_details
        rec = pywt.waverec2(new_coeff, 'db1')
        denoised_channels.append(rec)

    wavelet_out = np.stack(denoised_channels, axis=2)
    current_h, current_w = wavelet_out.shape[:2]
    if current_h != original_h or current_w != original_w:
        wavelet_out = wavelet_out[:original_h, :original_w, :]
    wavelet_out = np.clip(wavelet_out, 0, 255).astype(np.uint8)

    cnn = cv2.fastNlMeansDenoisingColored(
        wavelet_out, None,
        h=max(5, noise_std * 0.4),
        hColor=max(5, noise_std * 0.4),
        templateWindowSize=7,
        searchWindowSize=21
    )

    residual = wavelet_out.astype(np.float32) - cnn.astype(np.float32)
    residual = cv2.GaussianBlur(residual, (3,3), 0)
    beta     = 0.7 if noise_std < 15 else 0.6 if noise_std < 25 else 0.5
    enhanced = cnn.astype(np.float32) + beta * residual

    gray   = cv2.cvtColor(wavelet_out, cv2.COLOR_BGR2GRAY)
    edges  = cv2.Canny(gray, 40, 120).astype(np.float32) / 255.0
    edges  = cv2.GaussianBlur(edges, (5,5), 0)
    edges  = edges[:,:,None]

    sharpen_kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    sharp  = cv2.filter2D(enhanced.astype(np.uint8), -1, sharpen_kernel)
    final  = enhanced * (1 - edges) + sharp.astype(np.float32) * edges

    final  = cv2.bilateralFilter(final.astype(np.uint8), d=5,
                                  sigmaColor=25, sigmaSpace=25)
    return final

def apply_bayesian_filter(image, noise_var_rgb, win=5):
    pad    = win // 2
    img    = image.astype(np.float32)
    padded = np.pad(img, ((pad,pad),(pad,pad),(0,0)), 'reflect')
    out    = np.zeros_like(img)
    H, W   = img.shape[:2]

    for i in range(H):
        for j in range(W):
            patch  = padded[i:i+win, j:j+win, :]
            center = patch[win//2, win//2, :]
            pixels = patch.reshape(-1, 3)
            mu     = np.mean(pixels, 0)
            var    = np.maximum(np.var(pixels, 0), 1e-6)
            k      = np.clip(1 - (noise_var_rgb / var), 0, 1)
            out[i, j] = mu + k * (center - mu)

    return np.clip(out, 0, 255).astype(np.uint8)


# ─────────────────────────────────────────────────────────────
# MAIN PROCESSING LOOP
# ─────────────────────────────────────────────────────────────
for image_file in image_files:
    print("\nProcessing:", image_file)

    img = cv2.imread(os.path.join(dataset_folder, image_file))
    if img is None:
        continue

    name       = os.path.splitext(image_file)[0]
    out_folder = os.path.join(output_root, name)
    os.makedirs(out_folder, exist_ok=True)

    results = []

    for level in noise_levels:
        std = level * 30   # IMPORTANT

        lvl_folder = os.path.join(out_folder, f"noise_{level}")
        os.makedirs(lvl_folder, exist_ok=True)

        noisy = add_gaussian_noise(img, std)
        cv2.imwrite(os.path.join(lvl_folder, "noisy.png"), noisy)

        noise_var     = std ** 2
        noise_var_rgb = np.array([noise_var] * 3, np.float32)

        # ── run all filters ───────────────────────────────────
        mean_out    = mean_filter(noisy)
        median_out  = median_filter(noisy)
        gauss_out   = gaussian_filter(noisy)
        wien_out    = wiener_filter_rgb(noisy)
        bayes_out   = apply_bayesian_filter(noisy, noise_var_rgb)
        bshrink_out = adaptive_bayesian_shrinkage(noisy, std)   # ← NEW
        cnn_out     = cnn_denoise(noisy)
        dncnn_out   = dncnn_denoise(noisy)
        bcnn_out    = wavelet_bayesian_cnn_denoise(noisy, std)

        # ── save outputs ──────────────────────────────────────
        cv2.imwrite(os.path.join(lvl_folder, "mean.png"),             mean_out)
        cv2.imwrite(os.path.join(lvl_folder, "median.png"),           median_out)
        cv2.imwrite(os.path.join(lvl_folder, "gaussian.png"),         gauss_out)
        cv2.imwrite(os.path.join(lvl_folder, "wiener.png"),           wien_out)
        cv2.imwrite(os.path.join(lvl_folder, "bayesian.png"),         bayes_out)
        cv2.imwrite(os.path.join(lvl_folder, "bayesian_shrink.png"),  bshrink_out)  # ← NEW
        cv2.imwrite(os.path.join(lvl_folder, "cnn.png"),              cnn_out)
        cv2.imwrite(os.path.join(lvl_folder, "dncnn.png"),            dncnn_out)
        cv2.imwrite(os.path.join(lvl_folder, "bayesian_cnn.png"),     bcnn_out)

        # ── log metrics ───────────────────────────────────────
        def log(method_name, out):
            results.append([
                level, method_name,
                compute_psnr(img, out),
                compute_ssim_rgb(img, out),
                compute_rmse(img, out)
            ])

        log("Mean",              mean_out)
        log("Median",            median_out)
        log("Gaussian",          gauss_out)
        log("Wiener",            wien_out)
        log("Bayesian",          bayes_out)
        log("BayesianShrinkage", bshrink_out)   # ← NEW
        log("CNN",               cnn_out)
        log("DnCNN",             dncnn_out)
        log("BayesianCNN",       bcnn_out)

    df = pd.DataFrame(results, columns=["Noise", "Method", "PSNR", "SSIM", "RMSE"])
    df.to_excel(os.path.join(out_folder, "results.xlsx"), index=False)

    print("\nBest per noise:")
    for lvl in noise_levels:
        sub  = df[df["Noise"] == lvl]
        best = sub.loc[sub["PSNR"].idxmax()]
        print(f"  Noise {lvl} → {best['Method']} ({best['PSNR']:.2f} dB)")

print("\n✅ DONE SUCCESSFULLY")

Mounted at /content/drive
Images Found: ['leena.jpeg', 'lung.jpeg', 'parrots.jpeg', 'capsicum.jpeg', 'baboon.jpeg']

Processing: leena.jpeg

Best per noise:
  Noise 0.1 → Bayesian (39.34 dB)
  Noise 0.2 → Bayesian (34.99 dB)
  Noise 0.3 → Bayesian (32.44 dB)
  Noise 0.4 → Bayesian (30.74 dB)
  Noise 0.5 → Bayesian (29.39 dB)

Processing: lung.jpeg

Best per noise:
  Noise 0.1 → Bayesian (39.74 dB)
  Noise 0.2 → Median (36.10 dB)
  Noise 0.3 → Mean (34.41 dB)
  Noise 0.4 → Gaussian (33.13 dB)
  Noise 0.5 → Gaussian (32.13 dB)

Processing: parrots.jpeg

Best per noise:
  Noise 0.1 → Bayesian (39.72 dB)
  Noise 0.2 → Bayesian (35.62 dB)
  Noise 0.3 → Bayesian (33.29 dB)
  Noise 0.4 → Bayesian (31.76 dB)
  Noise 0.5 → Bayesian (30.61 dB)

Processing: capsicum.jpeg

Best per noise:
  Noise 0.1 → Bayesian (39.15 dB)
  Noise 0.2 → Bayesian (34.67 dB)
  Noise 0.3 → Bayesian (32.14 dB)
  Noise 0.4 → Bayesian (30.37 dB)
  Noise 0.5 → Bayesian (29.13 dB)

Processing: baboon.jpeg

Best per noise:
